<a href="https://colab.research.google.com/github/PochampellyDeekshitha/DeepLearningPractice/blob/main/DL_ASSIGNMENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🔗 Kaggle Link: https://www.kaggle.com/datasets/datamunge/sign-language-mnist Sign Language MNIST Dataset

In [1]:
import numpy as np
import pandas as pd

train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

# Normalize
X_train = X_train / 255.0
X_test = X_test / 255.0

# One-hot encoding
def one_hot(y, num_classes=25):
    one_hot_y = np.zeros((y.size, num_classes))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y

y_train = one_hot(y_train)
y_test = one_hot(y_test)

input_size = 784
hidden_size = 128
output_size = 25
lr = 0.01
epochs = 50

# Initialize weights
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return x > 0

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def cross_entropy(y_true, y_pred):
    m = y_true.shape[0]
    return -np.sum(y_true * np.log(y_pred + 1e-8)) / m

# Training (Batch Gradient Descent)

for epoch in range(epochs):

    # Forward pass
    Z1 = np.dot(X_train, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    # Loss
    loss = cross_entropy(y_train, A2)

    # Backward pass
    m = X_train.shape[0]

    dZ2 = A2 - y_train
    dW2 = (1/m) * np.dot(A1.T, dZ2)
    db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)

    dW1 = (1/m) * np.dot(X_train.T, dZ1)
    db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

    # Update weights
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

def predict(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)
    return np.argmax(A2, axis=1)

y_pred = predict(X_test)
y_true = np.argmax(y_test, axis=1)

accuracy = np.mean(y_pred == y_true)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

Epoch 0, Loss: 3.2195
Epoch 5, Loss: 3.2189
Epoch 10, Loss: 3.2182
Epoch 15, Loss: 3.2176
Epoch 20, Loss: 3.2170
Epoch 25, Loss: 3.2164
Epoch 30, Loss: 3.2159
Epoch 35, Loss: 3.2153
Epoch 40, Loss: 3.2148
Epoch 45, Loss: 3.2143
Test Accuracy: 3.29%


In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std


def one_hot(y, num_classes=25):
    one_hot_y = np.zeros((y.size, num_classes))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y

y_train = one_hot(y_train)
y_test_original = y_test # Store original labels for accuracy calculation
y_test = one_hot(y_test)

input_size = 784
h1 = 256
h2 = 128
output_size = 25

lr = 0.001
epochs = 100
batch_size = 64
dropout_rate = 0.2

W1 = np.random.randn(input_size, h1) * np.sqrt(2. / input_size)
b1 = np.zeros((1, h1))

W2 = np.random.randn(h1, h2) * np.sqrt(2. / h1)
b2 = np.zeros((1, h2))

W3 = np.random.randn(h2, output_size) * np.sqrt(2. / h2)
b3 = np.zeros((1, output_size))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)


def cross_entropy(y_true, y_pred):
    return -np.mean(np.sum(y_true * np.log(y_pred + 1e-8), axis=1))


for epoch in range(epochs):

    # Shuffle data
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    for i in range(0, X_train.shape[0], batch_size):

        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        D1 = (np.random.rand(*A1.shape) > dropout_rate) / (1 - dropout_rate)
        A1 *= D1

        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)

        D2 = (np.random.rand(*A2.shape) > dropout_rate) / (1 - dropout_rate)
        A2 *= D2

        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        m = X_batch.shape[0]

        dZ3 = A3 - y_batch
        dW3 = (1/m) * np.dot(A2.T, dZ3)
        db3 = (1/m) * np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, W3.T)
        dA2 *= D2
        dZ2 = dA2 * relu_derivative(Z2)

        dW2 = (1/m) * np.dot(A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dA1 *= D1
        dZ1 = dA1 * relu_derivative(Z1)

        dW1 = (1/m) * np.dot(X_batch.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        W1 -= lr * dW1
        b1 -= lr * db1

        W2 -= lr * dW2
        b2 -= lr * db2

        W3 -= lr * dW3
        b3 -= lr * db3

    if epoch % 10 == 0:
        Z1 = np.dot(X_train, W1) + b1
        A1 = relu(Z1)
        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)
        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        loss = cross_entropy(y_train, A3)
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

def predict(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = relu(Z2)
    Z3 = np.dot(A2, W3) + b3
    A3 = softmax(Z3)
    return np.argmax(A3, axis=1)

y_pred = predict(X_test)
accuracy = np.mean(y_pred == y_test_original)

print(f"\n Test Accuracy: {accuracy * 100:.2f}%")

Epoch 0, Loss: 2.7064
Epoch 10, Loss: 0.8521
Epoch 20, Loss: 0.4341
Epoch 30, Loss: 0.2497
Epoch 40, Loss: 0.1536
Epoch 50, Loss: 0.0992
Epoch 60, Loss: 0.0671
Epoch 70, Loss: 0.0476
Epoch 80, Loss: 0.0348
Epoch 90, Loss: 0.0264


In [ ]:
''' improve with lr'''
'''
initial_lr = 0.001
decay_rate = 0.95

for epoch in range(epochs):
    lr = initial_lr * (decay_rate ** epoch)  '''

decay_rate = 0.95

for epoch in range(epochs):

    # ---- Training loop ----
    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        # Forward pass
        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)

        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        # Backprop
        m = X_batch.shape[0]

        dZ3 = A3 - y_batch
        dW3 = (1/m) * np.dot(A2.T, dZ3)
        db3 = (1/m) * np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, W3.T)
        dZ2 = dA2 * relu_derivative(Z2)

        dW2 = (1/m) * np.dot(A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)

        dW1 = (1/m) * np.dot(X_batch.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        # Update
        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2
        W3 -= lr * dW3
        b3 -= lr * db3

    lr = lr * decay_rate

    if epoch % 10 == 0:
        print(f"Epoch {epoch}, LR: {lr:.6f}")

BGD showed smooth and stable convergence but required high computation per epoch. Training was slower due to full dataset usage, and accuracy was moderate. Compared to mini-batch GD, it had less generalization and higher time complexity.

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

X_train = X_train / 255.0
X_test = X_test / 255.0

def one_hot(y, num_classes=25):
    one_hot_y = np.zeros((y.size, num_classes))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y

y_train = one_hot(y_train)
y_test = one_hot(y_test)

input_size = 784
hidden_size = 128
output_size = 25

lr = 0.01
epochs = 20

W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
b2 = np.zeros((1, output_size))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x))
    return exp_x / np.sum(exp_x)

for epoch in range(epochs):
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    for i in range(X_train.shape[0]):
        x = X_train[i].reshape(1, -1)
        y = y_train[i].reshape(1, -1)

        Z1 = np.dot(x, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        dZ2 = A2 - y
        dW2 = np.dot(A1.T, dZ2)
        db2 = dZ2

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)

        dW1 = np.dot(x.T, dZ1)
        db1 = dZ1

        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    print(f"Epoch {epoch+1} completed")

def predict(X):
    preds = []
    for i in range(X.shape[0]):
        x = X[i].reshape(1, -1)
        Z1 = np.dot(x, W1) + b1
        A1 = relu(Z1)
        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)
        preds.append(np.argmax(A2))
    return np.array(preds)

y_pred = predict(X_test)
accuracy = np.mean(y_pred == y_test)

print(f"\n Test Accuracy (SGD): {accuracy * 100:.2f}%")

SGD updates weights after each sample, leading to faster but noisy convergence. It improves generalization but causes fluctuations and may not reach the exact minimum.

In [ ]:
import numpy as np
import pandas as pd

train = pd.read_csv("sign_mnist_train.csv")
test = pd.read_csv("sign_mnist_test.csv")

X_train = train.drop("label", axis=1).values
y_train = train["label"].values

X_test = test.drop("label", axis=1).values
y_test = test["label"].values

X_train = X_train / 255.0
X_test = X_test / 255.0

def one_hot(y, num_classes=25):
    one_hot_y = np.zeros((y.size, num_classes))
    one_hot_y[np.arange(y.size), y] = 1
    return one_hot_y

y_train = one_hot(y_train)
y_test = one_hot(y_test)

input_size = 784
hidden_size = 128
output_size = 25

lr = 0.01
epochs = 20
batch_size = 64

W1 = np.random.randn(input_size, hidden_size) * np.sqrt(2. / input_size)
b1 = np.zeros((1, hidden_size))

W2 = np.random.randn(hidden_size, output_size) * np.sqrt(2. / hidden_size)
b2 = np.zeros((1, output_size))

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

for epoch in range(epochs):
    indices = np.arange(X_train.shape[0])
    np.random.shuffle(indices)

    X_train = X_train[indices]
    y_train = y_train[indices]

    for i in range(0, X_train.shape[0], batch_size):
        X_batch = X_train[i:i+batch_size]
        y_batch = y_train[i:i+batch_size]

        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = softmax(Z2)

        m = X_batch.shape[0]

        dZ2 = A2 - y_batch
        dW2 = (1/m) * np.dot(A1.T, dZ2)
        db2 = (1/m) * np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)

        dW1 = (1/m) * np.dot(X_batch.T, dZ1)
        db1 = (1/m) * np.sum(dZ1, axis=0, keepdims=True)

        W1 -= lr * dW1
        b1 -= lr * db1
        W2 -= lr * dW2
        b2 -= lr * db2

    print(f"Epoch {epoch+1} completed")

def predict(X):
    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)
    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)
    return np.argmax(A2, axis=1)

y_pred = predict(X_test)
accuracy = np.mean(y_pred == y_test)

print(f"\nTest Accuracy (Mini-Batch GD): {accuracy * 100:.2f}%")

Mini-batch GD updates weights using small batches, offering faster and more stable convergence than BGD and SGD. It improves generalization and is widely used in practice.